In [2]:
# ============================================================
# 1. IMPORTS
# ============================================================

import os
import sys
import copy
from pathlib import Path

import numpy as np
import pandas as pd

from pyspark.sql import functions as F
from pyspark.ml.functions import vector_to_array

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
)

from xgboost.spark import SparkXGBClassifier

from credit_risk.utils.config import (
    read_config,
    create_path,
)

from credit_risk.utils.spark import create_spark_session

from credit_risk.modelling.artifacts_spark import (
    load_spark_model_artifacts,
)

from credit_risk.modelling.preprocessing_spark import (
    prepare_features_spark,
)

from credit_risk.modelling.data_spark import (
    load_modelling_vintage_spark,
)

In [3]:
# ============================================================
# 2. PROJECT SETUP
# ============================================================

project_path = Path.cwd().parent

if str(project_path / "src") not in sys.path:
    sys.path.insert(0, str(project_path / "src"))

os.chdir(project_path)

print("Project path:")
print(project_path)

Project path:
c:\Users\vorad\OneDrive\Desktop\Projects\mortgage-credit-risk


In [4]:
# ============================================================
# 3. LOAD CONFIGURATION
# ============================================================

config = read_config(project_path)

parameters = config["parameters"]

modelling_config = parameters["modelling"]
features_config = modelling_config["features"]

approach = parameters["modelling_approach"]
target = parameters["target"]["name"]

print("Modelling approach :", approach)
print("Engine             :", parameters["engine"])
print("Algorithm          :", modelling_config["algorithm"])
print("Model version      :", modelling_config["version"])
print("Target             :", target)

print()
print("Numerical features:")
print(features_config["numerical_features"])

print()
print("Categorical features:")
print(features_config["categorical_features"])

print()
print("Engineered features:")
print(features_config["engineered_features"])

Modelling approach : behavioral
Engine             : pyspark
Algorithm          : xgboost
Model version      : test_unbalanced_quarterly
Target             : future_90dpd_12m

Numerical features:
['number_of_borrowers', 'mi_percentage', 'original_upb', 'credit_score', 'original_dti', 'original_ltv', 'original_cltv', 'current_actual_upb', 'current_interest_rate', 'estimated_ltv', 'calculated_loan_age', 'remaining_months_to_legal_maturity', 'current_non_interest_bearing_upb', 'current_interest_bearing_upb', 'non_interest_bearing_upb_pct', 'interest_bearing_upb_pct', 'current_dpd_numeric', 'max_dpd_to_date', 'delinquency_months_to_date', 'months_since_last_delinquency', 'ever_30dpd_to_date', 'ever_60dpd_to_date', 'ever_modified', 'ever_payment_deferred', 'ever_borrower_assistance', 'ever_disaster_delinquency', 'dpd_30_count_6m', 'dpd_60_count_6m', 'max_dpd_6m', 'delinquency_months_6m', 'modification_count_6m', 'payment_deferral_count_6m', 'borrower_assistance_count_6m', 'disaster_delinque

In [5]:
# ============================================================
# 4. SPARK
# ============================================================

spark = create_spark_session(config)

print("Spark session created.")

c:\Users\vorad\OneDrive\Desktop\Projects\mortgage-credit-risk\venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Spark session created.


In [6]:
# ============================================================
# 5. LOAD BASELINE MODEL + PREPROCESSOR
# ============================================================

model, preprocessor = load_spark_model_artifacts(config)

print("Baseline model:")
print(model)

print()
print("Baseline preprocessor:")
print(preprocessor)

Baseline model:
SparkXGBClassifier_5b23668040cc

Baseline preprocessor:
PipelineModel_8f3738fd76e2


In [7]:
# ============================================================
# 6. BASELINE FEATURE SET
# ============================================================

X_columns = (
    features_config["numerical_features"]
    + features_config["categorical_features"]
    + features_config["engineered_features"]
)

print("Number of configured features:", len(X_columns))

print()
for i, feature in enumerate(X_columns, start=1):
    print(f"{i:2d}. {feature}")

Number of configured features: 61

 1. number_of_borrowers
 2. mi_percentage
 3. original_upb
 4. credit_score
 5. original_dti
 6. original_ltv
 7. original_cltv
 8. current_actual_upb
 9. current_interest_rate
10. estimated_ltv
11. calculated_loan_age
12. remaining_months_to_legal_maturity
13. current_non_interest_bearing_upb
14. current_interest_bearing_upb
15. non_interest_bearing_upb_pct
16. interest_bearing_upb_pct
17. current_dpd_numeric
18. max_dpd_to_date
19. delinquency_months_to_date
20. months_since_last_delinquency
21. ever_30dpd_to_date
22. ever_60dpd_to_date
23. ever_modified
24. ever_payment_deferred
25. ever_borrower_assistance
26. ever_disaster_delinquency
27. dpd_30_count_6m
28. dpd_60_count_6m
29. max_dpd_6m
30. delinquency_months_6m
31. modification_count_6m
32. payment_deferral_count_6m
33. borrower_assistance_count_6m
34. disaster_delinquency_count_6m
35. rate_step_count_6m
36. dpd_30_count_12m
37. dpd_60_count_12m
38. max_dpd_12m
39. delinquency_months_12m
40. m

In [8]:
# ============================================================
# 7. ABLATION DEFINITION
# ============================================================

ABLATION_NAME = "drop_3_weak_features"

ABLATION_DROP = [
    "first_time_homebuyer_flag",
    "mi_cancellation_indicator",
    "calculated_loan_age",
]

print("Ablation:", ABLATION_NAME)

print()
print("Features being removed:")

for feature in ABLATION_DROP:
    print(" -", feature)

Ablation: drop_3_weak_features

Features being removed:
 - first_time_homebuyer_flag
 - mi_cancellation_indicator
 - calculated_loan_age


In [9]:
# ============================================================
# 8. REDUCED FEATURE CONFIGURATION
# ============================================================

ablation_config = copy.deepcopy(config)

ablation_features_config = ablation_config["parameters"]["modelling"]["features"]

for key in [
    "numerical_features",
    "categorical_features",
    "engineered_features",
]:
    ablation_features_config[key] = [
        feature
        for feature in ablation_features_config[key]
        if feature not in ABLATION_DROP
    ]

X_columns_ablation = (
    ablation_features_config["numerical_features"]
    + ablation_features_config["categorical_features"]
    + ablation_features_config["engineered_features"]
)

print("Baseline feature count :", len(X_columns))
print("Ablation feature count :", len(X_columns_ablation))

print()
print("Remaining features:")

for i, feature in enumerate(X_columns_ablation, start=1):
    print(f"{i:2d}. {feature}")

Baseline feature count : 61
Ablation feature count : 58

Remaining features:
 1. number_of_borrowers
 2. mi_percentage
 3. original_upb
 4. credit_score
 5. original_dti
 6. original_ltv
 7. original_cltv
 8. current_actual_upb
 9. current_interest_rate
10. estimated_ltv
11. remaining_months_to_legal_maturity
12. current_non_interest_bearing_upb
13. current_interest_bearing_upb
14. non_interest_bearing_upb_pct
15. interest_bearing_upb_pct
16. current_dpd_numeric
17. max_dpd_to_date
18. delinquency_months_to_date
19. months_since_last_delinquency
20. ever_30dpd_to_date
21. ever_60dpd_to_date
22. ever_modified
23. ever_payment_deferred
24. ever_borrower_assistance
25. ever_disaster_delinquency
26. dpd_30_count_6m
27. dpd_60_count_6m
28. max_dpd_6m
29. delinquency_months_6m
30. modification_count_6m
31. payment_deferral_count_6m
32. borrower_assistance_count_6m
33. disaster_delinquency_count_6m
34. rate_step_count_6m
35. dpd_30_count_12m
36. dpd_60_count_12m
37. max_dpd_12m
38. delinquenc

In [10]:
# ============================================================
# 9. VALIDATE FEATURE REMOVAL
# ============================================================

unexpectedly_missing = [
    feature for feature in ABLATION_DROP if feature not in X_columns
]

if unexpectedly_missing:
    raise ValueError(
        "Requested ablation features were not present in "
        f"baseline feature set: {unexpectedly_missing}"
    )

if any(feature in X_columns_ablation for feature in ABLATION_DROP):
    raise RuntimeError(
        "Ablation failed. One or more dropped features " "are still present."
    )

print("Ablation validation passed.")

Ablation validation passed.


In [11]:
# ============================================================
# 10. LOAD TRAINING SPLIT
# ============================================================

train_path = create_path(
    config["catalog"]["base"],
    config["catalog"],
    "train_df",
    approach,
    must_exist=True,
)

print("Training split:")
print(train_path)

train_df = spark.read.parquet(str(train_path))

print()
print("Training rows:", f"{train_df.count():,}")
print("Training columns:", len(train_df.columns))

Training split:
data\04_model_split\behavioral\train_split.parquet

Training rows: 10,970,334
Training columns: 80


In [12]:
# ============================================================
# 11. BASELINE TRAINING INPUT
# ============================================================

X_train = train_df.select(*X_columns)

X_train_prepared = prepare_features_spark(
    X_train,
    config,
)

X_train_transformed = preprocessor.transform(
    X_train_prepared,
)

X_train_transformed.select("features").printSchema()

root
 |-- features: vector (nullable = true)



In [13]:
# ============================================================
# 12. ABLATION TRAINING INPUT
# ============================================================

X_train_ablation = train_df.select(*X_columns_ablation)

X_train_ablation_prepared = prepare_features_spark(
    X_train_ablation,
    ablation_config,
)

print("Ablation training input:")
X_train_ablation_prepared.printSchema()

Ablation training input:
root
 |-- number_of_borrowers: long (nullable = true)
 |-- mi_percentage: long (nullable = true)
 |-- original_upb: long (nullable = true)
 |-- credit_score: long (nullable = true)
 |-- original_dti: long (nullable = true)
 |-- original_ltv: long (nullable = true)
 |-- original_cltv: long (nullable = true)
 |-- current_actual_upb: double (nullable = true)
 |-- current_interest_rate: double (nullable = true)
 |-- estimated_ltv: long (nullable = true)
 |-- remaining_months_to_legal_maturity: long (nullable = true)
 |-- current_non_interest_bearing_upb: double (nullable = true)
 |-- current_interest_bearing_upb: double (nullable = true)
 |-- non_interest_bearing_upb_pct: double (nullable = true)
 |-- interest_bearing_upb_pct: double (nullable = true)
 |-- current_dpd_numeric: double (nullable = true)
 |-- max_dpd_to_date: double (nullable = true)
 |-- delinquency_months_to_date: short (nullable = true)
 |-- months_since_last_delinquency: double (nullable = false)


In [14]:
# ============================================================
# 13. BUILD ABLATION PREPROCESSOR
# ============================================================

import inspect
import credit_risk.modelling.preprocessing_spark as preprocessing_spark

print("Available preprocessing functions containing 'preprocess':")

for name in dir(preprocessing_spark):
    if "preprocess" in name.lower():
        print(" -", name)

Available preprocessing functions containing 'preprocess':
 - build_preprocessor_spark
 - fit_preprocessor_spark
 - transform_with_preprocessor_spark


In [15]:
ablation_preprocessor = preprocessing_spark.build_preprocessor_spark(ablation_config)

In [16]:
# ============================================================
# 14. FIT ABLATION PREPROCESSOR
# ============================================================

ablation_preprocessor_model = ablation_preprocessor.fit(X_train_ablation_prepared)

print(ablation_preprocessor_model)

ERROR:root:Exception while sending command.
Traceback (most recent call last):
  File "c:\Users\vorad\OneDrive\Desktop\Projects\mortgage-credit-risk\venv\Lib\site-packages\py4j\clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ~~~~~~~~~~~~~~~~~~~~^^
  File "C:\Users\vorad\AppData\Local\Programs\Python\Python313\Lib\socket.py", line 719, in readinto
    return self._sock.recv_into(b)
           ~~~~~~~~~~~~~~~~~~~~^^^
ConnectionResetError: [WinError 10054] An existing connection was forcibly closed by the remote host

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "c:\Users\vorad\OneDrive\Desktop\Projects\mortgage-credit-risk\venv\Lib\site-packages\py4j\java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "c:\Users\vorad\OneDrive\Desktop\Projects\mortgage-credit-risk\venv\Lib\site-packages\py4j\clientser

Py4JError: An error occurred while calling o1176.fit